# gufe framejs visualizations

This notebook demos the new interactive `.view()` integration on gufe objects.
It runs entirely on **gufe** (no `openfe` needed) — the visualization classes all
live here.

Rendering happens through [framejs.io](https://framejs.io): a canonical viz is
published once, and each object pushes its own data to that frame over the widget
comm channel (no URL size limit).

> Started with `just dev` → open me at **http://localhost:8888/lab**.

## Bare-cell auto-display

Leave a `LigandNetwork` as the last line of a cell and it renders itself — no
`.view()` call needed (via `_repr_mimebundle_`).

In [1]:
from pathlib import Path

import gufe

data = Path(gufe.__file__).parent / "tests" / "data"
net = gufe.LigandNetwork.from_graphml((data / "ligand_network.graphml").read_text())
net  # bare-cell → interactive framejs radial graph

/src/gufe/src/gufe/components/explicitmoleculecomponent.py:59: UserWarning: Molecule (name='') doesn't have any hydrogen atoms present. If this is unexpected, consider loading the molecule with `removeHs=False`
  warnings.warn(


## Explicit `.view()`

Same widget, called explicitly — accepts `width` / `height` overrides.

In [12]:
net.view(height="600px")
from IPython.display import HTML
HTML('''<div id="hm" style="font:12px monospace;white-space:pre;padding:6px;background:#eef;color:#000">measuring…</div>
<script>
setTimeout(()=>{
  const H=n=>n?Math.round(n.getBoundingClientRect().height):'—';
  const cs=[...document.querySelectorAll('.metaframe-widget-container')];
  let out = cs.length+' widget(s):\\n';
  cs.forEach((c,i)=>{
    // walk UP to the jupyter output area, recording inline heights
    let up=[], n=c;
    for(let d=0; d<6 && n; d++, n=n.parentElement){
      const ih=n.style.height||'';
      const cls=(n.className||'').toString().split(' ').find(x=>x.startsWith('jp-'))|| (n.style.height?'el':'');
      up.push(`${cls||'div'}=${H(n)}${ih?`(set ${ih})`:''}`);
    }
    const inner=c.querySelector(':scope>div'), grid=c.querySelector('div[style*="grid"]'), ifr=c.querySelector('iframe');
    out += `#${i}: UP[${up.join(' < ')}]  DOWN[container=${H(c)} inner=${H(inner)} grid=${H(grid)} iframe=${H(ifr)}]\\n`;
  });
  document.getElementById('hm').textContent=out;
},1500);
</script>''')

## The shareable CLI URL

The same viz on the command line: `build_cli_url` returns the canonical
framejs.io frame with this network appended as `#?inputs=<base64>`, so it opens in
a browser with no live kernel.

In [3]:
from gufe.visualization.framejs import build_cli_url

build_cli_url(net)

'https://framejs.io/#?js=...&og=JTdCJTIydGl0bGUlMjIlM0ElMjJndWZlJTIwTGlnYW5kTmV0d29yayUyMiUyQyUyMmRlc2NyaXB0aW9uJTIyJTNBJTIySW50ZXJhY3RpdmUlMjBmcmVlLWVuZXJneSUyMGxpZ2FuZCUyMG5ldHdvcmslM0ElMjBjbGljayUyMGFuJTIwZWRnZSUyMHRvJTIwZHJpdmUlMjBhJTIwM0QlMjBhdG9tLW1hcHBpbmclMjB2aWV3ZXIlMjBmb3IlMjBpdHMlMjB0d28lMjBlbmRwb2ludCUyMG1vbGVjdWxlcy4lMjIlN0Q=&modules=JTVCJTIyaHR0cHMlM0ElMkYlMkYzZG1vbC5vcmclMkZidWlsZCUyRjNEbW9sLW1pbi5qcyUyMiU1RA==&inputs=JTdCJTIybmV0d29yay5ncmFwaG1sJTIyJTNBJTIwJTIyJTNDZ3JhcGhtbCUyMHhtbG5zJTNEJTVDJTIyaHR0cCUzQSUyRiUyRmdyYXBobWwuZ3JhcGhkcmF3aW5nLm9yZyUyRnhtbG5zJTVDJTIyJTIweG1sbnMlM0F4c2klM0QlNUMlMjJodHRwJTNBJTJGJTJGd3d3LnczLm9yZyUyRjIwMDElMkZYTUxTY2hlbWEtaW5zdGFuY2UlNUMlMjIlMjB4c2klM0FzY2hlbWFMb2NhdGlvbiUzRCU1QyUyMmh0dHAlM0ElMkYlMkZncmFwaG1sLmdyYXBoZHJhd2luZy5vcmclMkZ4bWxucyUyMGh0dHAlM0ElMkYlMkZncmFwaG1sLmdyYXBoZHJhd2luZy5vcmclMkZ4bWxucyUyRjEuMCUyRmdyYXBobWwueHNkJTVDJTIyJTNFJTVDbiUyMCUyMCUzQ2tleSUyMGlkJTNEJTVDJTIyZDIlNUMlMjIlMjBmb3IlM0QlNUMlMjJlZGdlJTVDJTIyJTIwYXR0ci5uYW1lJTN

## Real OpenFE demo inputs (optional)

`just dev` also clones **openfe-demo** locally (git-ignored) at
`/workspace/openfe-demo`. Its `src/inputs/` has real ligand/protein files you can
load into gufe objects. The full `openfe_demo.ipynb` there needs the heavy
`openfe` env, but the input data is handy for exercising the gufe vizzes.

In [4]:
inputs = Path("/workspace/openfe-demo/src/inputs")
sorted(p.name for p in inputs.glob("*")) if inputs.exists() else "run `just dev` to clone openfe-demo"

['bace_ligands_nagl_1.sdf',
 'bace_ligands_nagl_2.sdf',
 'bace_protein.pdb',
 'settings_radial.yaml',
 'tyk2_ligands.sdf',
 'tyk2_ligands_charged.sdf',
 'tyk2_protein.pdb']

In [5]:
import metaframe_widget
from metaframe_widget import MetaframeWidget
print(metaframe_widget.__file__)
print("LEN", len(MetaframeWidget._esm), "ROOT", "const root = document.createElement" in MetaframeWidget._esm)
import json, urllib.request  # or just check the JupyterLab About/api


/src/metaframe-widget/src/metaframe_widget/__init__.py
LEN 8068 ROOT True


In [8]:
from IPython.display import display, HTML
w = net.view(height="600px")
display(w)
display(HTML('''<div id="hm2" style="font:12px monospace;white-space:pre;padding:6px;background:#efe;color:#000">measuring…</div>
<script>
setTimeout(()=>{
  const H=n=>n?Math.round(n.getBoundingClientRect().height):'—';
  const cs=[...document.querySelectorAll('.metaframe-widget-container')];
  const c=cs[cs.length-1];
  if(!c){document.getElementById('hm2').textContent='no widget';return;}
  let up=[],n=c;
  for(let d=0;d<6&&n;d++,n=n.parentElement){
    const ih=n.style.height||'';
    const j=(n.className||'').toString().split(' ').find(x=>x.startsWith('jp-'))||(ih?'el':'div');
    up.push(j+'='+H(n)+(ih?'(set '+ih+')':''));
  }
  const inner=c.querySelector(':scope>div'),grid=c.querySelector('div[style*="grid"]'),ifr=c.querySelector('iframe');
  document.getElementById('hm2').textContent='UP['+up.join(' < ')+']\\nDOWN[container='+H(c)+' inner='+H(inner)+' grid='+H(grid)+' iframe='+H(ifr)+']';
},2000);
</script>'''))

In [9]:
net.view(height="600px")


In [10]:
from IPython.display import HTML
HTML('''<div id="hm3" style="font:12px monospace;white-space:pre;padding:6px;background:#fee;color:#000">measuring…</div>
<script>
setTimeout(()=>{
  const H=n=>n?Math.round(n.getBoundingClientRect().height):'—';
  const cs=[...document.querySelectorAll('.metaframe-widget-container')];
  const c=cs[cs.length-1];
  if(!c){document.getElementById('hm3').textContent='no widget';return;}
  let up=[],n=c;
  for(let d=0;d<7&&n;d++,n=n.parentElement){
    const ih=n.style.height||'';const cs2=getComputedStyle(n);
    const j=(n.className||'').toString().split(' ').find(x=>x.startsWith('jp-'))||(ih?'el':'div');
    up.push(j+'='+H(n)+(ih?'(set '+ih+')':'')+(cs2.resize!=='none'?' RESIZE='+cs2.resize:''));
  }
  document.getElementById('hm3').textContent='UP['+up.join(' < ')+']';
},2000);
</script>''')